# DIVE-HMM · CCBF — Notebook 1 (Foundation + Core Method)

**CCBF = Calibrated Condition-Belief Filtering** — a lightweight, descriptor-agnostic probabilistic
layer for sequential Visual Place Recognition under extreme appearance change (seasons / weather / day-night).

### The research thesis (why this can be A*-level)
Prior sequential-VPR filters (FAB-MAP, HM4, Xu et al. 2021) plug **raw retrieval similarities** into a Bayes
filter with a **fixed, condition-agnostic observation model** and track **place only**. We argue the real
bottleneck in hard conditions is **miscalibration that *compounds* over the sequence**, not descriptor quality.

CCBF instead tracks the **appearance condition as a latent variable jointly with place**, and uses the
inferred condition to produce a **calibrated, condition-conditional observation likelihood**
`p(descriptor | place, condition)`. Consequences:
1. **Label-free online domain adaptation** — the condition belief is updated by pure Bayesian filtering (no labels, no gradients at test time).
2. **Better-calibrated place posteriors** — lower ECE, principled "lost / off-map" detection.
3. **Descriptor-agnostic** — it is a thin layer on top of *any* global descriptor (SALAD, DINOv2, BoQ, MixVPR…).

### What this notebook (NB1) does — Phases 0→5
- **P0** Environment & dataset validation (offline, Kaggle, bundle introspection).
- **P1** Data pipeline (Nordland / RobotCar loaders; numeric frame-ID ordering).
- **P2** Feature extraction + disk cache (SALAD → DINOv2 → placeholder fallback).
- **P3** Baselines (single-image retrieval; fixed-observation Bayes filter).
- **P4** **Core CCBF** (condition model, calibrated likelihood, factored (place, condition) filter, null state).
- **P5** Small-scale end-to-end validation on **Nordland Short** (smoke test) + checkpoint save/load.

> **How to use:** run top-to-bottom. `CONFIG.SMOKE_TEST = True` (default) runs on the tiny Nordland Short
> subset (3,500 images) so you can verify the whole pipeline in minutes. Flip to `False` only after smoke passes.
> Everything is designed for Kaggle **Internet OFF** using the offline `bundle`.

## Phase 0 — Environment & Dataset Validation

We (0.0) import & set seeds, (0.1) define a single `CONFIG`, (0.2) set offline env + wire the bundle into
`sys.path`, (0.3) **introspect the bundle** (so if model loading later fails we can see the real API), and
(0.4) **validate datasets** without trusting folder names at test time. A global `REPORT` collects pass/fail
for every step so the final summary tells you exactly what is healthy.

In [ ]:
# === P0.0 — Imports, device, reproducibility ===========================================
import os, sys, json, time, glob, math, hashlib, warnings, traceback
from dataclasses import dataclass, field, asdict
from collections import defaultdict, OrderedDict

import numpy as np

# REPORT: a single source of truth for what passed / failed in this run.
REPORT = OrderedDict()
def log(step, status, msg=""):
    REPORT[step] = {"status": status, "msg": msg}
    tag = {"ok": "[ OK ]", "warn": "[WARN]", "fail": "[FAIL]", "info": "[info]"}.get(status, "[????]")
    print(f"{tag} {step}: {msg}")

def _seed_everything(seed=0):
    import random
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception:
        pass

try:
    import torch
    _HAS_TORCH = True
except Exception as e:  # pragma: no cover - torch always present on Kaggle
    _HAS_TORCH = False
    print("WARNING: torch not importable yet:", e)

_seed_everything(0)

if _HAS_TORCH:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("torch", torch.__version__, "| device:", DEVICE)
    if DEVICE == "cuda":
        p = torch.cuda.get_device_properties(0)
        print(f"GPU: {p.name} | VRAM: {p.total_memory/1e9:.1f} GB")
        # BF16 is safe & fast on Blackwell (RTX PRO 6000); gate on capability.
        BF16_OK = torch.cuda.is_bf16_supported()
    else:
        BF16_OK = False
    log("P0.0.torch", "ok", f"{torch.__version__} on {DEVICE} (bf16={BF16_OK})")
else:
    DEVICE = "cpu"; BF16_OK = False
    log("P0.0.torch", "fail", "torch not importable")
print("numpy", np.__version__)

In [ ]:
# === P0.1 — CONFIG (single place to control everything) ================================
@dataclass
class Config:
    # --- run mode ---
    SMOKE_TEST: bool = True          # True => tiny subset, fast. Flip to False for full scale.
    SEED: int = 0

    # --- Kaggle roots to search for datasets & bundle (auto-discovery handles layout) ---
    INPUT_ROOTS: tuple = ("/kaggle/input",)
    WORKING_DIR: str = "/kaggle/working/dive_hmm"
    CACHE_SUBDIR: str = "features"
    CKPT_SUBDIR: str = "ckpt"

    # --- image preprocessing ---
    IMG_SIZE: int = 322              # multiple of 14 for DINOv2 ViT-B/14 (23*14). 224 also valid.
    RESIZE_SHORTER: bool = False     # False => square resize to (IMG_SIZE, IMG_SIZE)

    # --- extraction ---
    BATCH_SIZE: int = 64             # raised automatically on big-VRAM GPUs below
    NUM_WORKERS: int = 2
    USE_BF16: bool = True

    # --- smoke-test sizing (per season) ---
    SMOKE_MAX_FRAMES: int = 600      # cap frames/season in smoke so a run takes minutes

    # --- CCBF model ---
    K_CONDITIONS: int = 2            # latent appearance conditions (e.g. summer/winter -> >=2)
    USE_TRAIN_CONDITION_LABELS: bool = True  # fit calibration with known train labels; infer at test
    STICKY_RHO: float = 0.97         # P(cond stays) — condition drifts slowly
    MOTION_STD: float = 2.0          # std (in place-index units) of the place motion kernel
    MOTION_VELOCITY: float = 1.0     # expected places advanced per step (same-route traversal)
    MOTION_LEAK: float = 1e-3        # uniform leak in the place transition
    NULL_PRIOR: float = 0.02         # prior mass on the off-map/null place state
    NULL_SELF: float = 0.9           # null self-transition probability

    # --- evaluation ---
    RECALL_KS: tuple = (1, 5, 10)
    PLACE_TOL: int = 2               # a prediction is correct if within +/- this many frame-indices
    ECE_BINS: int = 15

    @property
    def cache_dir(self): return os.path.join(self.WORKING_DIR, self.CACHE_SUBDIR)
    @property
    def ckpt_dir(self):  return os.path.join(self.WORKING_DIR, self.CKPT_SUBDIR)

CONFIG = Config()

# Exploit large VRAM when available (never trade correctness for it).
if _HAS_TORCH and DEVICE == "cuda":
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if total_gb >= 70:
        CONFIG.BATCH_SIZE = 256
    elif total_gb >= 40:
        CONFIG.BATCH_SIZE = 128
CONFIG.USE_BF16 = CONFIG.USE_BF16 and BF16_OK

os.makedirs(CONFIG.cache_dir, exist_ok=True)
os.makedirs(CONFIG.ckpt_dir, exist_ok=True)
print(json.dumps({k: v for k, v in asdict(CONFIG).items()}, indent=2, default=str))
log("P0.1.config", "ok", f"SMOKE_TEST={CONFIG.SMOKE_TEST}, batch={CONFIG.BATCH_SIZE}, bf16={CONFIG.USE_BF16}")

In [ ]:
# === P0.2 — Offline environment + path discovery =======================================
# Force every library into offline mode (Kaggle Internet OFF; the bundle provides assets).
for k, v in {
    "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "HF_DATASETS_OFFLINE": "1",
    "PIP_NO_INDEX": "1", "WANDB_MODE": "offline", "TOKENIZERS_PARALLELISM": "false",
}.items():
    os.environ.setdefault(k, v)

def _find_dir(markers_any, markers_all=(), roots=None, max_depth=4):
    """Breadth-first search under roots for a directory that contains all `markers_all`
    sub-names and at least one of `markers_any`. Returns the first match or None."""
    roots = roots or CONFIG.INPUT_ROOTS
    seen = set()
    for root in roots:
        if not os.path.isdir(root):
            continue
        frontier = [(root, 0)]
        while frontier:
            d, depth = frontier.pop(0)
            if d in seen or depth > max_depth:
                continue
            seen.add(d)
            try:
                entries = set(os.listdir(d))
            except Exception:
                continue
            ok_all = all(any(m.lower() in e.lower() for e in entries) for m in markers_all) if markers_all else True
            ok_any = any(any(m.lower() in e.lower() for e in entries) for m in markers_any) if markers_any else True
            if ok_all and ok_any:
                return d
            for e in sorted(entries):
                sub = os.path.join(d, e)
                if os.path.isdir(sub):
                    frontier.append((sub, depth + 1))
    return None

PATHS = {}
PATHS["bundle"]         = _find_dir(markers_any=["checkpoints"], markers_all=["checkpoints", "sources"])
PATHS["nordland_short"] = _find_dir(markers_any=["summer", "winter"], markers_all=["summer", "winter"])
PATHS["nordland_full"]  = _find_dir(markers_any=["data"], markers_all=["data", "annotations"])
PATHS["robotcar"]       = _find_dir(markers_any=["images"], markers_all=["images", "3D-models"])
PATHS["cmu"]            = _find_dir(markers_any=["slice2", "slice3", "slice6"])

for name, p in PATHS.items():
    if p:
        log(f"P0.2.path.{name}", "ok", p)
    else:
        log(f"P0.2.path.{name}", "warn", "not found (ok if you don't need it in NB1)")

# Wire bundle source code into sys.path so vendored SALAD/DINOv2/HLoc import offline.
if PATHS["bundle"]:
    srcroot = os.path.join(PATHS["bundle"], "sources")
    for sub in ["salad", "dinov2", "Hierarchical-Localization", "LightGlue", ""]:
        p = os.path.join(srcroot, sub) if sub else srcroot
        if os.path.isdir(p) and p not in sys.path:
            sys.path.insert(0, p)
    log("P0.2.syspath", "ok", f"added bundle sources from {srcroot}")
else:
    log("P0.2.syspath", "warn", "bundle not found; model loading will use fallbacks")

In [ ]:
# === P0.3 — Bundle introspection (so we can SEE the real API if loading fails) =========
def _tree(root, max_entries=40, max_depth=3):
    out = []
    root = os.path.abspath(root)
    for cur, dirs, files in os.walk(root):
        depth = cur[len(root):].count(os.sep)
        if depth > max_depth:
            dirs[:] = []
            continue
        rel = os.path.relpath(cur, root)
        indent = "  " * depth
        out.append(f"{indent}{'' if rel == '.' else os.path.basename(cur) + '/'}")
        for f in sorted(files)[:max_entries]:
            out.append(f"{indent}  {f}")
        dirs.sort()
    return "\n".join(out[:300])

if PATHS["bundle"]:
    b = PATHS["bundle"]
    ck = os.path.join(b, "checkpoints")
    if os.path.isdir(ck):
        print("checkpoints/:", sorted(os.listdir(ck)))
    # Show the salad / dinov2 source layout — key for adapting the loader to YOUR bundle.
    for sub in ["sources/salad", "sources/dinov2"]:
        p = os.path.join(b, sub)
        if os.path.isdir(p):
            print(f"\n--- {sub} ---")
            print(_tree(p, max_entries=20, max_depth=2))
    log("P0.3.bundle", "ok", "introspected (checkpoints + salad/dinov2 sources printed above)")
else:
    log("P0.3.bundle", "warn", "no bundle to introspect")

In [ ]:
# === P0.4 — Dataset validation =========================================================
def _list_images(d, exts=(".png", ".jpg", ".jpeg")):
    if not d or not os.path.isdir(d):
        return []
    return [f for f in os.listdir(d) if f.lower().endswith(exts)]

def _frame_id(fname):
    """Extract the numeric frame id from names like images-00251.png. Returns int or None."""
    digits = "".join(ch for ch in os.path.splitext(fname)[0] if ch.isdigit())
    return int(digits) if digits else None

def _find_season_dir(root, season):
    """Nordland Short nests as <root>/nordland-part-2020/<season>/; Full as <root>/data/<season>/<season>/."""
    cands = []
    for cur, dirs, files in os.walk(root):
        base = os.path.basename(cur).lower()
        if base == season and any(f.lower().endswith((".png", ".jpg")) for f in files):
            cands.append(cur)
    # prefer the deepest (handles data/winter/winter/)
    return sorted(cands, key=lambda p: -p.count(os.sep))[0] if cands else None

NORDLAND_ROOT = PATHS["nordland_short"] if CONFIG.SMOKE_TEST else (PATHS["nordland_full"] or PATHS["nordland_short"])
NORDLAND_SEASONS = ["summer", "winter"] if (CONFIG.SMOKE_TEST or not PATHS["nordland_full"]) else ["spring", "summer", "fall", "winter"]

season_dirs, season_counts = {}, {}
if NORDLAND_ROOT:
    for s in NORDLAND_SEASONS:
        sd = _find_season_dir(NORDLAND_ROOT, s)
        if sd:
            imgs = _list_images(sd)
            season_dirs[s] = sd
            season_counts[s] = len(imgs)
    print("Nordland root:", NORDLAND_ROOT)
    for s in NORDLAND_SEASONS:
        print(f"  {s:7s}: {season_counts.get(s, 0):6d} imgs  @ {season_dirs.get(s, 'MISSING')}")

    # Cross-season frame-ID synchronization check (the key correspondence signal).
    if len(season_dirs) >= 2:
        id_sets = {s: set(_frame_id(f) for f in _list_images(d)) - {None} for s, d in season_dirs.items()}
        common = set.intersection(*id_sets.values())
        union = set.union(*id_sets.values())
        jacc = len(common) / max(1, len(union))
        print(f"  shared frame-IDs across {list(season_dirs)}: {len(common)} (Jaccard={jacc:.3f})")
        log("P0.4.nordland", "ok" if jacc > 0.9 else "warn",
            f"{len(season_dirs)} seasons, {len(common)} synced frame-IDs (Jaccard={jacc:.3f})")
    else:
        log("P0.4.nordland", "warn", f"only {len(season_dirs)} season dir(s) found")
else:
    log("P0.4.nordland", "fail", "no Nordland root found — NB1 smoke needs Nordland Short")

# Dark-frame flagging (flag, never delete — may be a valid low-information location).
def dark_frame_fraction(image_dir, sample=60, thr_mean=12.0):
    try:
        from PIL import Image
    except Exception:
        return None
    files = sorted(_list_images(image_dir))[:sample]
    if not files:
        return None
    dark = 0
    for f in files:
        try:
            arr = np.asarray(Image.open(os.path.join(image_dir, f)).convert("L"), dtype=np.float32)
            if arr.mean() < thr_mean:
                dark += 1
        except Exception:
            pass
    return dark / max(1, len(files))

if season_dirs:
    s0 = list(season_dirs)[0]
    frac = dark_frame_fraction(season_dirs[s0])
    if frac is not None:
        log("P0.4.darkframes", "ok", f"~{frac*100:.1f}% near-black in sampled '{s0}' (flagged, not deleted)")

## Phase 1 — Data pipeline

A **Track** is one ordered traversal (a season/condition) = a list of `(frame_id, path)` sorted by the
**numeric** frame id (never lexical). For Nordland, two tracks that share a frame id depict the **same place**
under different conditions — this exact-filename correspondence is our ground truth. Condition labels come from
the folder name and are used **only at train time**; the test-time filter never reads them.

In [ ]:
# === P1.1 — Track abstraction & ground-truth correspondence ============================
class Track:
    """One ordered traversal under a single (train-time) condition label."""
    def __init__(self, name, condition, image_dir):
        self.name = name
        self.condition = condition           # train-time label only
        self.image_dir = image_dir
        files = [f for f in os.listdir(image_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))]
        pairs = [(_frame_id(f), f) for f in files]
        pairs = [(fid, f) for fid, f in pairs if fid is not None]
        pairs.sort(key=lambda x: x[0])       # numeric ordering
        self.frame_ids = [fid for fid, _ in pairs]
        self.paths = [os.path.join(image_dir, f) for _, f in pairs]

    def __len__(self): return len(self.paths)

    def subsample(self, max_frames):
        """Evenly subsample to <= max_frames while preserving order & frame-id spacing."""
        n = len(self.paths)
        if max_frames is None or n <= max_frames:
            return
        idx = np.linspace(0, n - 1, max_frames).round().astype(int)
        idx = sorted(set(idx.tolist()))
        self.frame_ids = [self.frame_ids[i] for i in idx]
        self.paths = [self.paths[i] for i in idx]

def _restrict_to_ids(track, keep_ids):
    pos = {fid: j for j, fid in enumerate(track.frame_ids)}
    idx = [pos[f] for f in keep_ids if f in pos]
    track.frame_ids = [track.frame_ids[j] for j in idx]
    track.paths = [track.paths[j] for j in idx]

def build_nordland_tracks():
    """Build season tracks ROW-ALIGNED on the common frame-id grid, so row i depicts the
    same place across every season (clean GT; place-motion velocity == 1)."""
    raw = {}
    for s in NORDLAND_SEASONS:
        if s in season_dirs:
            t = Track(name=f"nordland_{s}", condition=s, image_dir=season_dirs[s])
            if len(t) > 0:
                raw[s] = t
    if len(raw) < 2:
        return raw
    common = sorted(set.intersection(*[set(t.frame_ids) for t in raw.values()]))
    if CONFIG.SMOKE_TEST and len(common) > CONFIG.SMOKE_MAX_FRAMES:
        sel = np.linspace(0, len(common) - 1, CONFIG.SMOKE_MAX_FRAMES).round().astype(int)
        common = [common[i] for i in sorted(set(sel.tolist()))]
    for t in raw.values():
        _restrict_to_ids(t, common)
    return raw

def gt_match_index(query_track, ref_track):
    """For each query frame, the index into ref_track with the same frame id (or -1 if absent)."""
    ref_pos = {fid: j for j, fid in enumerate(ref_track.frame_ids)}
    return np.array([ref_pos.get(fid, -1) for fid in query_track.frame_ids], dtype=np.int64)

TRACKS = build_nordland_tracks()
print("Tracks:", {s: len(t) for s, t in TRACKS.items()})
if len(TRACKS) >= 2:
    a, b = list(TRACKS)[0], list(TRACKS)[1]
    gt = gt_match_index(TRACKS[a], TRACKS[b])
    log("P1.1.tracks", "ok", f"{len(TRACKS)} tracks; GT coverage {a}->{b}: {(gt>=0).mean()*100:.1f}%")
else:
    log("P1.1.tracks", "fail", "need >=2 Nordland tracks for NB1 smoke")

In [ ]:
# === P1.2 — Image dataset & transforms =================================================
from PIL import Image
try:
    import torchvision.transforms as T
    _IMAGENET_MEAN = (0.485, 0.456, 0.406); _IMAGENET_STD = (0.229, 0.224, 0.225)
    def build_transform(size):
        return T.Compose([T.Resize((size, size), antialias=True),
                          T.ToTensor(),
                          T.Normalize(_IMAGENET_MEAN, _IMAGENET_STD)])
    _HAS_TV = True
except Exception:
    _HAS_TV = False
    def build_transform(size):
        mean = np.array([0.485, 0.456, 0.406], np.float32); std = np.array([0.229, 0.224, 0.225], np.float32)
        def _tx(img):
            img = img.resize((size, size)).convert("RGB")
            arr = (np.asarray(img, np.float32) / 255.0 - mean) / std
            return torch.from_numpy(arr.transpose(2, 0, 1).copy())
        return _tx

class ImageList(torch.utils.data.Dataset):
    def __init__(self, paths, size):
        self.paths = paths; self.tx = build_transform(size)
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        try:
            img = Image.open(self.paths[i]).convert("RGB")
        except Exception:
            img = Image.new("RGB", (self.tx_size if hasattr(self, "tx_size") else 322,) * 2)
        return self.tx(img), i

if not _HAS_TV:
    CONFIG.NUM_WORKERS = 0            # closure transform isn't picklable for worker processes
log("P1.2.transform", "ok", f"img_size={CONFIG.IMG_SIZE}, torchvision={_HAS_TV}, workers={CONFIG.NUM_WORKERS}")

## Phase 2 — Feature extraction + disk cache

Global descriptors are produced by a **pluggable extractor** with graceful fallback:
`SALAD` (from the bundle) → `DINOv2-GeM` (vendored ViT-B/14) → `placeholder` (hand-crafted, always works).
CCBF is descriptor-agnostic, so the whole pipeline — filter, calibration, metrics — is validated regardless
of which extractor loads. Descriptors are L2-normalized, cached to disk, and the cache is resumable.

In [ ]:
# === P2.1 — Pluggable feature extractor (SALAD -> DINOv2-GeM -> placeholder) ===========
EXTRACTOR_INFO = {"name": None, "dim": None, "detail": ""}

def _gem_pool(x, p=3.0, eps=1e-6):           # x: (B, N, C) patch tokens
    x = x.clamp(min=eps).pow(p).mean(dim=1).pow(1.0 / p)
    return x

def _try_build_salad():
    """Load SALAD (serizba/salad, CVPR'24) from the bundle. We build backbone+aggregator directly via
    `helper.get_*` with the released dino_salad.ckpt configs, bypassing VPRModel (so no PyTorch-Lightning /
    pytorch_metric_learning deps). dim = token_dim + num_clusters*cluster_dim = 256 + 64*128 = 8448."""
    if not PATHS["bundle"]:
        return None
    ck = os.path.join(PATHS["bundle"], "checkpoints", "dino_salad.ckpt")
    salad_dir = os.path.join(PATHS["bundle"], "sources", "salad")
    if not (os.path.isfile(ck) and os.path.isdir(salad_dir)):
        return None
    try:
        if salad_dir not in sys.path:
            sys.path.insert(0, salad_dir)
        from models import helper                                   # salad/models/helper.py
        backbone = helper.get_backbone("dinov2_vitb14",
            {"num_trainable_blocks": 4, "return_token": True, "norm_layer": True})
        aggregator = helper.get_aggregator("SALAD",
            {"num_channels": 768, "num_clusters": 64, "cluster_dim": 128, "token_dim": 256})
        sd = torch.load(ck, map_location="cpu"); sd = sd.get("state_dict", sd)
        bb = {k[len("backbone."):]: v for k, v in sd.items() if k.startswith("backbone.")}
        ag = {k[len("aggregator."):]: v for k, v in sd.items() if k.startswith("aggregator.")}
        mb, ub = backbone.load_state_dict(bb, strict=False)
        ma, ua = aggregator.load_state_dict(ag, strict=False)
        print(f"  SALAD: backbone(missing={len(mb)},unexpected={len(ub)}) agg(missing={len(ma)},unexpected={len(ua)})")
        model = torch.nn.Module(); model.backbone = backbone; model.aggregator = aggregator
        model.eval().to(DEVICE)
        @torch.no_grad()
        def fwd(x):
            return model.aggregator(model.backbone(x))
        with torch.no_grad():
            d = fwd(torch.zeros(2, 3, CONFIG.IMG_SIZE, CONFIG.IMG_SIZE, device=DEVICE))
        if not torch.isfinite(d).all() or d.ndim != 2 or d.shape[1] < 1000:
            print("  SALAD: sanity forward failed, dim", tuple(d.shape)); return None
        return fwd, d.shape[1]
    except Exception:
        import traceback; print("  SALAD error:"); traceback.print_exc(); return None

def _try_build_dinov2():
    """Vendored DINOv2 ViT-B/14 + GeM pooling of patch tokens. Returns (fwd, dim) or None."""
    pth = os.path.join(PATHS["bundle"], "checkpoints", "dinov2_vitb14_pretrain.pth") if PATHS["bundle"] else None
    vit = None
    for import_fn in (
        lambda: __import__("dinov2.models.vision_transformer", fromlist=["vit_base"]).vit_base,
    ):
        try:
            vit_base = import_fn()
            vit = vit_base(patch_size=14, img_size=518, init_values=1.0, block_chunks=0)
            break
        except Exception as e:
            print("  DINOv2: vit_base build failed ->", repr(e)[:140])
    if vit is None:
        return None
    try:
        if pth and os.path.isfile(pth):
            sd = torch.load(pth, map_location="cpu"); sd = sd.get("state_dict", sd)
            m, u = vit.load_state_dict(sd, strict=False)
            print(f"  DINOv2: weights loaded (missing={len(m)}, unexpected={len(u)})")
        else:
            print("  DINOv2: WARNING pretrain .pth not found; random init (degraded)")
        vit.eval().to(DEVICE)
        @torch.no_grad()
        def fwd(x):
            feats = vit.forward_features(x)
            # DINOv2 tokens are SIGNED (LayerNorm output) -> use CLS + signed mean pooling.
            # (GeM/clamp would discard the negative half-space and collapse descriptors to a narrow cone.)
            if isinstance(feats, dict):
                parts = []
                cls = feats.get("x_norm_clstoken")
                pt = feats.get("x_norm_patchtokens")
                if cls is not None: parts.append(cls)
                if pt is not None:  parts.append(pt.mean(dim=1))
                g = torch.cat(parts, dim=1) if parts else feats
            else:
                g = feats.mean(dim=1) if feats.dim() == 3 else feats
            return g
        with torch.no_grad():
            d = fwd(torch.zeros(2, 3, CONFIG.IMG_SIZE, CONFIG.IMG_SIZE, device=DEVICE))
        if not torch.isfinite(d).all():
            print("  DINOv2: non-finite forward"); return None
        return fwd, d.shape[1]
    except Exception as e:
        print("  DINOv2: error ->", repr(e)[:160]); return None

def _build_placeholder():
    """Dependency-free global descriptor (tiled RGB+gradient histogram). DEGRADED: pipeline validation only."""
    GRID = 4; BINS = 8
    @torch.no_grad()
    def fwd(x):                               # x: (B,3,H,W) normalized; undo roughly for stats
        B, C, H, W = x.shape
        gh, gw = H // GRID, W // GRID
        feats = []
        for i in range(GRID):
            for j in range(GRID):
                tile = x[:, :, i*gh:(i+1)*gh, j*gw:(j+1)*gw]
                feats.append(tile.mean(dim=(2, 3)))                    # (B,3) mean color
                feats.append(tile.std(dim=(2, 3)))                     # (B,3) contrast
                gx = (tile[:, :, :, 1:] - tile[:, :, :, :-1]).abs().mean(dim=(2, 3))
                feats.append(gx)                                       # (B,3) gradient energy
        return torch.cat(feats, dim=1)
    dim = GRID * GRID * 9
    return fwd, dim

def build_extractor():
    order = [("salad", _try_build_salad), ("dinov2", _try_build_dinov2)]
    for name, builder in order:
        if PATHS["bundle"]:
            print(f"Trying extractor: {name} ...")
            res = builder()
            if res is not None:
                fwd, dim = res
                EXTRACTOR_INFO.update(name=name, dim=dim, detail="loaded from bundle")
                log("P2.1.extractor", "ok", f"{name} (dim={dim})")
                return fwd
    fwd, dim = _build_placeholder()
    EXTRACTOR_INFO.update(name="placeholder", dim=dim, detail="DEGRADED — real models unavailable")
    log("P2.1.extractor", "warn", f"placeholder descriptor (dim={dim}) — results NOT publication-grade")
    return fwd

EXTRACT_FWD = build_extractor()

In [ ]:
# === P2.2 — Batched extraction + resumable disk cache ==================================
def _cache_path(tag): return os.path.join(CONFIG.cache_dir, f"{tag}.npy")
def _manifest_path(tag): return os.path.join(CONFIG.cache_dir, f"{tag}.json")

@torch.no_grad()
def extract_track(track, tag, force=False):
    """Return (N, D) float32 L2-normalized descriptors for a Track, using a resumable cache."""
    cp, mp = _cache_path(tag), _manifest_path(tag)
    if (not force) and os.path.isfile(cp) and os.path.isfile(mp):
        man = json.load(open(mp))
        if man.get("extractor") == EXTRACTOR_INFO["name"] and man.get("n") == len(track) \
           and man.get("frame_ids") == track.frame_ids:
            arr = np.load(cp)
            print(f"  cache hit: {tag} -> {arr.shape}")
            return arr.astype(np.float32)

    loader = torch.utils.data.DataLoader(
        ImageList(track.paths, CONFIG.IMG_SIZE), batch_size=CONFIG.BATCH_SIZE,
        shuffle=False, num_workers=CONFIG.NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
    feats = np.empty((len(track), EXTRACTOR_INFO["dim"]), dtype=np.float32)
    amp_dtype = torch.bfloat16 if CONFIG.USE_BF16 else torch.float16
    use_amp = (DEVICE == "cuda")
    t0 = time.time()
    for imgs, idx in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        if use_amp:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                d = EXTRACT_FWD(imgs)
        else:
            d = EXTRACT_FWD(imgs)
        d = torch.nn.functional.normalize(d.float(), dim=1)   # L2-normalize in fp32
        feats[idx.numpy()] = d.detach().cpu().numpy()
    dt = time.time() - t0
    np.save(cp, feats)
    json.dump({"extractor": EXTRACTOR_INFO["name"], "dim": int(EXTRACTOR_INFO["dim"]),
               "n": len(track), "frame_ids": track.frame_ids}, open(mp, "w"))
    print(f"  extracted {tag}: {feats.shape} in {dt:.1f}s ({len(track)/max(dt,1e-6):.1f} img/s)")
    assert np.isfinite(feats).all(), f"non-finite descriptors in {tag}"
    return feats

DESC = {}
for s, t in TRACKS.items():
    DESC[s] = extract_track(t, tag=f"nordland_{'smoke' if CONFIG.SMOKE_TEST else 'full'}_{s}")
if DESC:
    dims = {s: v.shape for s, v in DESC.items()}
    log("P2.2.cache", "ok", f"descriptors cached: {dims}")
else:
    log("P2.2.cache", "fail", "no descriptors extracted")

## Phase 3 — Baselines

Two baselines to beat:
1. **Single-image retrieval** — argmax cosine similarity per frame (no temporal reasoning).
2. **Fixed-observation Bayes filter** — the classic sequential-VPR recipe (FAB-MAP / HM4 / Xu'21 style):
   a place-only HMM with a band **motion model** and a **single, condition-agnostic temperature** on the
   similarities. This is exactly what CCBF upgrades by making the observation model condition-conditional
   and calibrated.

We also define the shared evaluation tools: `recall@K` with a place tolerance, and **ECE** (Expected
Calibration Error) — the headline reliability metric.

In [ ]:
# === P3.0 — Shared evaluation utilities ===============================================
def cosine_sim(Q, R):
    """(T,D)x(M,D) -> (T,M). Inputs assumed L2-normalized; we renormalize defensively."""
    Q = Q / (np.linalg.norm(Q, axis=1, keepdims=True) + 1e-12)
    R = R / (np.linalg.norm(R, axis=1, keepdims=True) + 1e-12)
    return Q @ R.T

def _logsumexp(x, axis=None):
    x = np.asarray(x, dtype=np.float64)
    m = np.max(x, axis=axis, keepdims=True)
    s = m + np.log(np.sum(np.exp(x - m), axis=axis, keepdims=True))
    if axis is None:
        return float(s.reshape(()))       # numpy 2.x: float() needs a 0-d array
    return np.squeeze(s, axis=axis)

def recall_at_k(sim, q_ids, r_ids, ks=(1, 5, 10), tol=2):
    """Fraction of queries whose top-k retrieved places include one within +/- tol frame-ids."""
    r_ids = np.asarray(r_ids); q_ids = np.asarray(q_ids)
    kmax = min(max(ks), sim.shape[1])
    topk = np.argsort(-sim, axis=1)[:, :kmax]
    out = {}
    for k in ks:
        kk = min(k, sim.shape[1])
        hit = np.abs(r_ids[topk[:, :kk]] - q_ids[:, None]) <= tol   # (T,kk)
        out[k] = float(hit.any(axis=1).mean())
    return out

def ece_from_conf(conf, correct, n_bins=15):
    """Expected Calibration Error + per-bin stats. conf,correct are 1D arrays over queries."""
    conf = np.asarray(conf, float); correct = np.asarray(correct, float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece, rows = 0.0, []
    N = len(conf)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        m = (conf > lo) & (conf <= hi) if i > 0 else (conf >= lo) & (conf <= hi)
        if m.sum() == 0:
            continue
        c_mean, a_mean, w = conf[m].mean(), correct[m].mean(), m.sum() / N
        ece += w * abs(a_mean - c_mean)
        rows.append((0.5 * (lo + hi), c_mean, a_mean, int(m.sum())))
    return float(ece), rows

def pred_correct(pred_idx, q_ids, r_ids, tol=2):
    r_ids = np.asarray(r_ids); q_ids = np.asarray(q_ids)
    return (np.abs(r_ids[pred_idx] - q_ids) <= tol).astype(float)

def fit_temperature(sim, gt_idx, grid=None):
    """Pick tau minimizing NLL of the GT place under softmax(sim/tau). gt_idx: (T,), -1 to skip."""
    grid = grid if grid is not None else np.logspace(-2.2, 0.3, 30)
    valid = gt_idx >= 0
    s, g = sim[valid], gt_idx[valid]
    best_tau, best_nll = grid[0], np.inf
    for tau in grid:
        logp = s / tau
        logp = logp - _logsumexp(logp, axis=1)[:, None]
        nll = -logp[np.arange(len(g)), g].mean()
        if nll < best_nll:
            best_nll, best_tau = nll, tau
    return float(best_tau), float(best_nll)

RESULTS = OrderedDict()   # method_name -> metrics dict

In [ ]:
# === P3.1 — Episode setup + single-image retrieval baseline ===========================
# Episode: reference MAP = one season, QUERY = another (hard cross-season).
# Calibration (train) vs evaluation split is interleaved by route index (honest-ish for NB1 smoke;
# NB2 will use a held-out condition). Both tracks are row-aligned, so GT is exact.
def build_episode(ref_season, query_season):
    R = DESC[ref_season]; Q = DESC[query_season]
    r_ids = np.asarray(TRACKS[ref_season].frame_ids)
    q_ids = np.asarray(TRACKS[query_season].frame_ids)
    sim = cosine_sim(Q, R)                                  # (T, M)
    gt = gt_match_index(TRACKS[query_season], TRACKS[ref_season])  # (T,) exact row->row
    T = sim.shape[0]
    train_mask = (np.arange(T) % 2 == 0) & (gt >= 0)
    eval_mask  = (np.arange(T) % 2 == 1) & (gt >= 0)
    return dict(sim=sim, gt=gt, r_ids=r_ids, q_ids=q_ids,
                train_mask=train_mask, eval_mask=eval_mask,
                ref_season=ref_season, query_season=query_season)

def eval_single_image(ep):
    sim, gt = ep["sim"], ep["gt"]
    tau, _ = fit_temperature(sim[ep["train_mask"]], gt[ep["train_mask"]])
    ev = ep["eval_mask"]
    rec = recall_at_k(sim[ev], ep["q_ids"][ev], ep["r_ids"], CONFIG.RECALL_KS, CONFIG.PLACE_TOL)
    # confidence = calibrated softmax max; correctness = top-1 within tol
    logp = sim[ev] / tau; prob = np.exp(logp - _logsumexp(logp, axis=1)[:, None])
    pred = np.argmax(prob, axis=1); conf = prob.max(axis=1)
    corr = pred_correct(pred, ep["q_ids"][ev], ep["r_ids"], CONFIG.PLACE_TOL)
    ece, _ = ece_from_conf(conf, corr, CONFIG.ECE_BINS)
    return {**rec, "ECE": ece, "acc": float(corr.mean()), "tau": tau}

if len(TRACKS) >= 2 and len(DESC) >= 2:
    REF_SEASON, QUERY_SEASON = "summer", ("winter" if "winter" in DESC else list(DESC)[-1])
    if REF_SEASON not in DESC:
        REF_SEASON = list(DESC)[0]
    EPISODE = build_episode(REF_SEASON, QUERY_SEASON)
    RESULTS["single_image"] = eval_single_image(EPISODE)
    print(f"Episode: ref={REF_SEASON}  query={QUERY_SEASON}  (T={EPISODE['sim'].shape[0]}, M={EPISODE['sim'].shape[1]})")
    print("single_image:", {k: round(v, 4) for k, v in RESULTS["single_image"].items()})
    log("P3.1.single", "ok", f"R@1={RESULTS['single_image'][1]:.3f}, ECE={RESULTS['single_image']['ECE']:.3f}")
else:
    log("P3.1.single", "fail", "need two seasons with descriptors")

In [ ]:
# === P3.2 — Fixed-observation Bayes filter (place-only, condition-agnostic) ============
def make_motion_kernel(velocity, std):
    lo = int(math.floor(velocity - 4 * std)); hi = int(math.ceil(velocity + 4 * std))
    offs = np.arange(lo, hi + 1)
    w = np.exp(-0.5 * ((offs - velocity) / max(std, 1e-3)) ** 2)
    w = w / w.sum()
    return offs, w

def predict_place(place, offs, w, leak):
    """1-D band motion: new[j] = sum_o w[o]*place[j-o], + uniform leak. Mass off the ends is dropped."""
    M = place.shape[0]; new = np.zeros_like(place)
    for off, wt in zip(offs, w):
        if off == 0:
            new += wt * place
        elif off > 0:
            if off < M: new[off:] += wt * place[:M - off]
        else:
            a = -off
            if a < M: new[:M - a] += wt * place[a:]
    s = new.sum()
    if s > 0: new /= s
    new = (1 - leak) * new + leak / M
    return new

def run_place_filter(sim, tau, q_ids=None, null_quantile=0.98, exit_prob=0.01):
    """Forward Bayesian filter over M places + 1 null state with a FIXED (single-tau) observation model.
    Returns (pred_idx[T], conf[T], null_post[T]) using the real-place posterior."""
    T, M = sim.shape
    offs, w = make_motion_kernel(CONFIG.MOTION_VELOCITY, CONFIG.MOTION_STD)
    place = np.full(M, (1 - CONFIG.NULL_PRIOR) / M); null = CONFIG.NULL_PRIOR
    preds = np.empty(T, np.int64); confs = np.empty(T); nulls = np.empty(T)
    for t in range(T):
        # --- predict ---
        pred_place = (1 - exit_prob) * predict_place(place, offs, w, CONFIG.MOTION_LEAK) \
                     + null * (1 - CONFIG.NULL_SELF) / M
        pred_null = CONFIG.NULL_SELF * null + exit_prob * place.sum()
        # --- observation likelihood (log) ---
        ll_place = sim[t] / tau
        ll_null = np.quantile(sim[t], null_quantile) / tau
        logb = np.empty(M + 1)
        logb[:M] = np.log(pred_place + 1e-30) + ll_place
        logb[M] = np.log(pred_null + 1e-30) + ll_null
        logb -= _logsumexp(logb)
        b = np.exp(logb)
        place, null = b[:M], b[M]
        s = place.sum() + null
        place, null = place / s, null / s
        preds[t] = int(np.argmax(place)); confs[t] = float(place.max()); nulls[t] = float(null)
    return preds, confs, nulls

def eval_place_filter(ep):
    tau, _ = fit_temperature(ep["sim"][ep["train_mask"]], ep["gt"][ep["train_mask"]])
    preds, confs, nulls = run_place_filter(ep["sim"], tau, ep["q_ids"])
    ev = ep["eval_mask"]
    # recall@K from the filtered posterior would need full posteriors; report R@1 from argmax here,
    # and reuse retrieval top-K for K>1 ranking context.
    corr = pred_correct(preds[ev], ep["q_ids"][ev], ep["r_ids"], CONFIG.PLACE_TOL)
    ece, _ = ece_from_conf(confs[ev], corr, CONFIG.ECE_BINS)
    out = {1: float(corr.mean()), "ECE": ece, "acc": float(corr.mean()), "tau": tau}
    return out

if "EPISODE" in globals():
    RESULTS["fixed_filter"] = eval_place_filter(EPISODE)
    print("fixed_filter:", {k: round(v, 4) for k, v in RESULTS["fixed_filter"].items()})
    log("P3.2.fixed", "ok", f"R@1={RESULTS['fixed_filter'][1]:.3f}, ECE={RESULTS['fixed_filter']['ECE']:.3f}")
else:
    log("P3.2.fixed", "warn", "no episode to run the fixed filter on")

## Phase 4 — Core method: Calibrated Condition-Belief Filtering (CCBF)

CCBF runs a **factored Bayes filter** over `(place p_t, condition c_t)`:

- **Condition `c_t`** is a latent appearance state (K clusters). Its belief `b(c_t)` is updated online from the
  **visual descriptor only** (responsibility to condition centroids) + a **sticky** temporal prior
  (`P(c_{t+1}=c_t)=ρ`, because appearance drifts slowly). *This is label-free online domain inference.*
- **Place `p_t`** uses a band **motion model** and a **condition-conditional, calibrated** observation
  likelihood: `p(d_t | p_t=j) = Σ_k b(c_t=k) · exp(sim[t,j]/τ_k)`, where `τ_k` is a per-condition temperature
  fit to be calibrated. In easy conditions `τ_k` is small (sharp); in hard conditions it is large (humble) —
  which is exactly what prevents the miscalibration that a single global τ propagates over the sequence.
- A **null / off-map** state gives principled "lost" detection.

Mean-field factorization keeps it `O(M·K)` per step — negligible on top of descriptor extraction.
Removing the condition latent (K=1) collapses CCBF back to the fixed-observation baseline — a built-in ablation.

In [ ]:
# === P4.1 — Condition model (centroids + per-condition calibrated temperature) =========
def _pairwise_sq_dists(X, C):
    """(T,D),(K,D) -> (T,K) squared L2 distances via ||x||^2+||c||^2-2x·c (memory-safe)."""
    xn = (X * X).sum(1)[:, None]; cn = (C * C).sum(1)[None, :]
    return np.maximum(xn + cn - 2.0 * (X @ C.T), 0.0)

def fit_condition_model():
    """Returns dict(names, C(K,D), sigma, tau(K,)). Labels mode uses per-season mean descriptors as
    centroids; otherwise KMeans. Per-condition tau is fit against a DIFFERENT season (non-degenerate)."""
    names = list(DESC.keys())
    if CONFIG.USE_TRAIN_CONDITION_LABELS and len(names) >= 2:
        C = np.stack([DESC[s].mean(0) for s in names]).astype(np.float32)
    else:
        # unsupervised KMeans (Lloyd) on a pooled sample
        pool = np.concatenate([DESC[s] for s in names], 0)
        K = max(2, CONFIG.K_CONDITIONS)
        rng = np.random.default_rng(CONFIG.SEED)
        C = pool[rng.choice(len(pool), K, replace=False)].copy()
        for _ in range(25):
            a = np.argmin(_pairwise_sq_dists(pool, C), 1)
            newC = np.stack([pool[a == k].mean(0) if (a == k).any() else C[k] for k in range(K)])
            if np.allclose(newC, C, atol=1e-5): C = newC; break
            C = newC
        names = [f"cond{k}" for k in range(K)]
    # sigma: typical distance of points to their nearest centroid (controls responsibility sharpness)
    pool = np.concatenate([DESC[s] for s in DESC], 0)
    d2 = _pairwise_sq_dists(pool, C).min(1)
    sigma = float(np.sqrt(np.median(d2)) + 1e-6)
    # per-condition temperature, fit against a different season to avoid identical-image degeneracy
    tau = []
    season_list = list(DESC.keys())
    for i, s in enumerate(names):
        src = season_list[i] if s in DESC else season_list[i % len(season_list)]
        others = [o for o in season_list if o != src] or [src]
        ref = others[0]
        sims = cosine_sim(DESC[src], DESC[ref])
        gt = gt_match_index(TRACKS[src], TRACKS[ref]) if src in TRACKS and ref in TRACKS else \
             np.arange(sims.shape[0])
        tr = (np.arange(sims.shape[0]) % 2 == 0) & (gt >= 0)
        if tr.sum() < 5: tr = (gt >= 0)
        tau_s, _ = fit_temperature(sims[tr], gt[tr])
        tau.append(tau_s)
    COND = dict(names=names, C=C, sigma=sigma, tau=np.asarray(tau, np.float32))
    return COND

if len(DESC) >= 2:
    COND = fit_condition_model()
    print("Conditions:", COND["names"])
    print("  tau per condition:", {n: round(float(t), 4) for n, t in zip(COND["names"], COND["tau"])})
    print(f"  sigma={COND['sigma']:.4f}, centroids={COND['C'].shape}")
    log("P4.1.condmodel", "ok", f"K={len(COND['names'])}, tau={np.round(COND['tau'],3).tolist()}")
else:
    COND = None
    log("P4.1.condmodel", "fail", "need >=2 seasons to build a condition model")

In [ ]:
# === P4.2 — Condition responsibilities (visual-only; used at test time) ================
def cond_log_resp(desc, COND):
    """(T,D) -> (T,K) log appearance-responsibility to condition centroids. No labels used here."""
    d2 = _pairwise_sq_dists(desc.astype(np.float32), COND["C"])
    logr = -0.5 * d2 / (COND["sigma"] ** 2)
    logr -= _logsumexp(logr, axis=1)[:, None]
    return logr

def _sticky_predict(bc, rho, K):
    return rho * bc + (1.0 - rho) * (1.0 - bc) / max(K - 1, 1)

In [ ]:
# === P4.3 — The CCBF filter ============================================================
def run_ccbf(sim, query_desc, COND, null_quantile=0.98, exit_prob=0.01, return_traj=False):
    """Factored (place, condition) Bayes filter. Returns preds, confs, null_post, and (optional)
    the condition-belief trajectory bc_traj (T,K)."""
    T, M = sim.shape; K = len(COND["tau"])
    inv_tau = (1.0 / np.maximum(COND["tau"], 1e-4)).astype(np.float64)          # (K,)
    appear_logr = cond_log_resp(query_desc, COND)                               # (T,K)
    offs, w = make_motion_kernel(CONFIG.MOTION_VELOCITY, CONFIG.MOTION_STD)

    place = np.full(M, (1 - CONFIG.NULL_PRIOR) / M); null = CONFIG.NULL_PRIOR
    bc = np.full(K, 1.0 / K)
    preds = np.empty(T, np.int64); confs = np.empty(T); nulls = np.empty(T)
    bc_traj = np.empty((T, K)) if return_traj else None

    for t in range(T):
        # --- condition update (sticky prior x visual appearance evidence) ---
        bc = _sticky_predict(bc, CONFIG.STICKY_RHO, K)
        logbc = np.log(bc + 1e-30) + appear_logr[t]
        logbc -= _logsumexp(logbc); bc = np.exp(logbc)

        # --- place predict ---
        pred_place = (1 - exit_prob) * predict_place(place, offs, w, CONFIG.MOTION_LEAK) \
                     + null * (1 - CONFIG.NULL_SELF) / M
        pred_null = CONFIG.NULL_SELF * null + exit_prob * place.sum()

        # --- condition-mixture, calibrated observation likelihood (log) ---
        # ll_place[j] = logsumexp_k( log b_c[k] + sim[t,j] * inv_tau[k] )
        A = sim[t][None, :] * inv_tau[:, None] + np.log(bc + 1e-30)[:, None]     # (K,M)
        ll_place = _logsumexp(A, axis=0)                                        # (M,)
        snull = np.quantile(sim[t], null_quantile)
        ll_null = _logsumexp(np.log(bc + 1e-30) + snull * inv_tau)              # scalar

        logb = np.empty(M + 1)
        logb[:M] = np.log(pred_place + 1e-30) + ll_place
        logb[M] = np.log(pred_null + 1e-30) + ll_null
        logb -= _logsumexp(logb)
        b = np.exp(logb)
        place, null = b[:M], b[M]
        s = place.sum() + null; place, null = place / s, null / s

        preds[t] = int(np.argmax(place)); confs[t] = float(place.max()); nulls[t] = float(null)
        if return_traj: bc_traj[t] = bc
    return (preds, confs, nulls, bc_traj) if return_traj else (preds, confs, nulls)

In [ ]:
# === P4.4 — Evaluate CCBF on the episode ==============================================
def eval_ccbf(ep, COND):
    q_desc = DESC[ep["query_season"]]
    preds, confs, nulls = run_ccbf(ep["sim"], q_desc, COND)
    ev = ep["eval_mask"]
    corr = pred_correct(preds[ev], ep["q_ids"][ev], ep["r_ids"], CONFIG.PLACE_TOL)
    ece, _ = ece_from_conf(confs[ev], corr, CONFIG.ECE_BINS)
    return {1: float(corr.mean()), "ECE": ece, "acc": float(corr.mean())}

# Ablation: K=1 (no condition latent) must collapse toward the fixed-observation filter.
def eval_ccbf_k1(ep):
    q_desc = DESC[ep["query_season"]]
    tau1, _ = fit_temperature(ep["sim"][ep["train_mask"]], ep["gt"][ep["train_mask"]])
    COND1 = dict(names=["all"], C=q_desc.mean(0, keepdims=True).astype(np.float32),
                 sigma=1.0, tau=np.asarray([tau1], np.float32))
    preds, confs, nulls = run_ccbf(ep["sim"], q_desc, COND1)
    ev = ep["eval_mask"]
    corr = pred_correct(preds[ev], ep["q_ids"][ev], ep["r_ids"], CONFIG.PLACE_TOL)
    ece, _ = ece_from_conf(confs[ev], corr, CONFIG.ECE_BINS)
    return {1: float(corr.mean()), "ECE": ece, "acc": float(corr.mean())}

if "EPISODE" in globals() and COND is not None:
    RESULTS["ccbf"] = eval_ccbf(EPISODE, COND)
    RESULTS["ccbf_K1_ablation"] = eval_ccbf_k1(EPISODE)
    print("ccbf:         ", {k: round(v, 4) for k, v in RESULTS["ccbf"].items()})
    print("ccbf_K1 (abl):", {k: round(v, 4) for k, v in RESULTS["ccbf_K1_ablation"].items()})
    log("P4.4.ccbf", "ok", f"R@1={RESULTS['ccbf'][1]:.3f}, ECE={RESULTS['ccbf']['ECE']:.3f}")
else:
    log("P4.4.ccbf", "warn", "CCBF not evaluated (missing episode or condition model)")

## Phase 5 — Small-scale end-to-end validation (smoke test)

We now (5.1) compare all methods on the cross-season episode, (5.2) demonstrate **label-free condition
tracking** on a synthesized condition-drift query, (5.3) verify **checkpoint save/load** reproduces outputs,
and (5.4) print a health gate. On the tiny smoke subset the *numbers* are only indicative — the goal here is
that the whole pipeline runs cleanly and the pieces behave sensibly before scaling up.

In [ ]:
# === P5.1 — Method comparison table ===================================================
def print_comparison(results):
    hdr = f"{'method':<20} {'R@1':>7} {'R@5':>7} {'R@10':>7} {'acc':>7} {'ECE':>7}"
    print(hdr); print("-" * len(hdr))
    for name, m in results.items():
        def g(k): return f"{m[k]:.3f}" if k in m else "   -  "
        print(f"{name:<20} {g(1):>7} {g(5):>7} {g(10):>7} {g('acc'):>7} {g('ECE'):>7}")

if RESULTS:
    print_comparison(RESULTS)
    # Sanity assertions for the smoke run (structural, not scientific thresholds).
    assert "ccbf" in RESULTS, "CCBF did not run"
    for name, m in RESULTS.items():
        for k, v in m.items():
            assert np.isfinite(v), f"non-finite metric {name}.{k}"
    if "fixed_filter" in RESULTS and "ccbf" in RESULTS:
        d_ece = RESULTS["fixed_filter"]["ECE"] - RESULTS["ccbf"]["ECE"]
        print(f"\nECE(fixed) - ECE(ccbf) = {d_ece:+.4f}  (positive => CCBF better calibrated)")
    log("P5.1.compare", "ok", "comparison table printed; all metrics finite")
else:
    log("P5.1.compare", "fail", "no results to compare")

In [ ]:
# === P5.2 — Label-free condition tracking (condition-drift demo) ======================
def condition_drift_demo():
    if not ("EPISODE" in globals() and COND is not None and len(DESC) >= 2):
        log("P5.2.drift", "warn", "skipped (need episode + condition model)"); return
    names = COND["names"]
    A, B = EPISODE["ref_season"], EPISODE["query_season"]
    if A not in names or B not in names:
        log("P5.2.drift", "warn", f"condition names {names} don't map to seasons {A}/{B}; skipped"); return
    descA, descB = DESC[A], DESC[B]
    M = descA.shape[0]; H = M // 2
    qdesc = np.concatenate([descA[:H], descB[H:]], 0)                 # appearance drifts A -> B
    true_cond = np.array([names.index(A)] * H + [names.index(B)] * (M - H))
    sim_drift = cosine_sim(qdesc, descA)
    preds, confs, nulls, bc = run_ccbf(sim_drift, qdesc, COND, return_traj=True)
    cond_pred = bc.argmax(1)
    acc = float((cond_pred == true_cond).mean())
    # first index in the 2nd half where the belief has switched to B
    switched = np.where(cond_pred[H:] == names.index(B))[0]
    cp = int(H + switched[0]) if len(switched) else -1
    print(f"Condition-tracking accuracy (label-free): {acc:.3f}  (chance={1/len(names):.3f})")
    print(f"True changepoint at t={H}; CCBF belief switches to '{B}' at t={cp}")
    try:
        import matplotlib; import matplotlib.pyplot as plt
        fig, ax = plt.subplots(2, 1, figsize=(9, 4.5), sharex=True)
        ax[0].plot(bc[:, names.index(B)], label=f"b(c={B})", color="tab:blue")
        ax[0].axvline(H, color="k", ls="--", lw=1, label="true drift")
        ax[0].set_ylabel("P(condition)"); ax[0].legend(loc="center left"); ax[0].set_ylim(-0.05, 1.05)
        ax[1].plot(confs, color="tab:green"); ax[1].set_ylabel("place conf"); ax[1].set_xlabel("frame t")
        fig.suptitle("CCBF label-free condition tracking on a drift query")
        out = os.path.join(CONFIG.WORKING_DIR, "p5_condition_drift.png")
        fig.tight_layout(); fig.savefig(out, dpi=110); plt.show()
        print("saved:", out)
    except Exception as e:
        print("(plot skipped:", repr(e)[:80], ")")
    log("P5.2.drift", "ok" if acc > 1.0 / len(names) else "warn",
        f"tracking acc={acc:.3f} vs chance={1/len(names):.3f}")

condition_drift_demo()

In [ ]:
# === P5.3 — Checkpoint save / load reproducibility ====================================
def test_checkpoint_roundtrip():
    if not ("EPISODE" in globals() and COND is not None):
        log("P5.3.ckpt", "warn", "skipped (no model)"); return
    path = os.path.join(CONFIG.ckpt_dir, "ccbf_nb1.pt")
    payload = dict(
        cond=dict(names=COND["names"], C=COND["C"], sigma=float(COND["sigma"]), tau=COND["tau"]),
        config=asdict(CONFIG), extractor=EXTRACTOR_INFO, results=RESULTS,
        episode=dict(ref=EPISODE["ref_season"], query=EPISODE["query_season"]))
    torch.save(payload, path)
    try:
        loaded = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        loaded = torch.load(path, map_location="cpu")
    COND2 = loaded["cond"]
    n = min(80, EPISODE["sim"].shape[0])
    qd = DESC[EPISODE["query_season"]][:n]
    p1, _, _ = run_ccbf(EPISODE["sim"][:n], qd, COND)
    p2, _, _ = run_ccbf(EPISODE["sim"][:n], qd, COND2)
    ok = bool(np.array_equal(p1, p2))
    sz = os.path.getsize(path) / 1e6
    print(f"checkpoint: {path} ({sz:.2f} MB); reload-identical predictions: {ok}")
    assert ok, "checkpoint round-trip changed predictions"
    log("P5.3.ckpt", "ok", f"saved {sz:.2f} MB, round-trip identical")

test_checkpoint_roundtrip()

In [ ]:
# === P5.4 — Health gate & next steps ==================================================
def summarize():
    n_ok = sum(1 for v in REPORT.values() if v["status"] == "ok")
    n_warn = sum(1 for v in REPORT.values() if v["status"] == "warn")
    n_fail = sum(1 for v in REPORT.values() if v["status"] == "fail")
    print(f"REPORT: {n_ok} ok / {n_warn} warn / {n_fail} fail")
    for step, v in REPORT.items():
        if v["status"] in ("warn", "fail"):
            print(f"  [{v['status']}] {step}: {v['msg']}")
    print()
    extractor = EXTRACTOR_INFO.get("name")
    smoke_pass = (n_fail == 0) and ("ccbf" in RESULTS)
    print("=" * 64)
    print("SMOKE TEST:", "PASS [OK]" if smoke_pass else "NEEDS ATTENTION [check warns/fails above]")
    print("Extractor used:", extractor, "(placeholder => load real SALAD/DINOv2 before trusting numbers)")
    print("=" * 64)
    print("""
Next steps:
  1. If extractor == 'placeholder' or 'dinov2' fell back: paste the P0.3 bundle introspection
     output + the P2.1 loader logs so the SALAD loader can be matched to your exact bundle API.
  2. If SMOKE PASS with a real descriptor: set CONFIG.SMOKE_TEST=False to cache full Nordland,
     then we move to NB2 (RobotCar night/rain/snow + CMU; full ECE / reliability / off-map AUROC).
  3. NB2 will add: real multi-condition episodes, open-set off-map injection, and the
     miscalibration-propagation (ECE-vs-sequence-length) theory experiment.
""")
    return smoke_pass

SMOKE_PASS = summarize()

### What you just ran
- A complete, offline, single-file CCBF pipeline: env/data validation → descriptor cache → baselines →
  **core CCBF** (factored place+condition filter with calibrated, condition-conditional likelihood) →
  smoke validation (comparison, **label-free condition tracking**, checkpoint round-trip).

### How to report back for the next iteration
Paste: (a) the **P0.3** bundle introspection, (b) the **P2.1** extractor log (which path loaded),
(c) the **P5.1** comparison table, and (d) the **P5.4** health gate. That tells us whether SALAD loaded
and whether CCBF already shows the expected calibration edge — and drives NB2.

> Reminder: on Nordland the smoke numbers are only a sanity signal (Nordland is "easy" for recall).
> The paper's headline — hard conditions (RobotCar/CMU), calibration/ECE, and open-set/off-map — comes in NB2.